# Tarification commune et validation

**Prime pure = probabilité de sinistre × coût attendu conditionnel.** Ce document réunit les deux composantes et distingue recherche, validation imbriquée, diagnostic historique et score Kaggle.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'run.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
from pricing.experiment import load_data
train, test, development, historique = load_data()
R = ROOT / 'reports'
selection = json.loads((R / 'selection.json').read_text(encoding='utf-8'))
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})


## Protocole

1. Partition par client : 80 % développement et 20 % diagnostic historique déjà connu.
2. Recherche sur cinq plis groupés, identiques pour fréquence et gravité.
3. Mélanges convexes prédéfinis et choix sur MSE de prime hors pli.
4. Validation imbriquée : trois plis externes ; dans chaque apprentissage externe, la recherche entière est répétée sur trois plis internes.
5. Réentraînement final sur tous les contrats / tous les sinistres positifs.

MSE et RMSE ordonnent les modèles de la même manière. Les scores de recherche sont optimistes après sélection. Le développement a déjà été exploré avant cette étude ; la validation imbriquée ne supprime pas cette limite historique.

In [2]:
pd.read_csv(R/'search_pairs.csv').head(12)

,frequency,severity,MSE,RMSE,fi,gi
0,0.75*cat_frequency_d6+0.25*reference,gamma_basic_a0.01,302572.363273,550.065781,31,6
1,0.75*cat_frequency_d6+0.25*reference,0.5*gamma_basic_a0.01+0.5*reference,302574.653529,550.067863,31,5
2,0.75*cat_frequency_d6+0.25*reference,gamma_basic_a0.001,302574.663749,550.067872,31,4
3,0.75*cat_frequency_d6+0.25*reference,0.5*gamma_basic_a0.001+0.5*reference,302574.830120,550.068023,31,3
4,0.75*cat_frequency_d6+0.25*reference,0.5*lgb_severity_regression+0.5*reference,302577.032835,550.070025,31,25
5,0.75*cat_frequency_d6+0.25*reference,0.5*cat_severity_d2+0.5*reference,302578.705957,550.071546,31,17
6,0.75*cat_frequency_d6+0.25*reference,gamma_reference,302579.682491,550.072434,31,0
7,0.75*cat_frequency_d5+0.25*reference,gamma_basic_a0.01,302583.613866,550.076007,27,6
8,0.75*cat_frequency_d6+0.25*reference,0.5*cat_severity_d4+0.5*reference,302583.891749,550.076260,31,21
9,0.75*cat_frequency_d5+0.25*reference,0.5*gamma_basic_a0.01+0.5*reference,302586.022146,550.078196,27,5


In [3]:
nested=json.loads((R/'nested_validation.json').read_text(encoding='utf-8'))
pd.DataFrame(nested['scores']).T

,MSE,RMSE,MAE,bias,predicted_observed_ratio
selected,302890.788558,550.355148,191.302134,-9.200744e-01,0.991133
reference,302925.011283,550.386238,191.848150,-1.834155e-01,0.998232
previous,302957.273601,550.415546,191.649710,-3.329233e-01,0.996792
constant,306411.975265,553.544917,195.333000,1.598184e-07,1.000000


In [4]:
pd.read_csv(R/'nested_folds.csv')

,fold,frequency,severity,MSE,RMSE,MAE,bias,predicted_observed_ratio,reference_RMSE,previous_RMSE
0,1,0.5*cat_frequency_d3+0.5*reference,0.5*cat_severity_d2+0.5*reference,326440.645685,571.349845,190.043067,1.627089,1.015968,571.340028,571.326578
1,2,0.5*cat_frequency_d5+0.5*reference,0.5*lgb_severity_gamma+0.5*reference,303129.501544,550.571977,191.444715,-4.700959,0.955494,550.541761,550.550839
2,3,0.5*logistic_splines_C0.3+0.5*reference,gamma_basic_a1.0,279100.434793,528.299569,192.418705,0.313739,1.003023,528.438833,528.535491


In [5]:
pd.DataFrame({k:nested[k] for k in ['gain_vs_reference','gain_vs_previous']}).T

,gain_RMSE,interval_95,repetitions,interpretation
gain_vs_reference,0.031091,"[-0.11742901761546136, 0.19099694249242982]",2000,Descriptive paired client bootstrap conditiona...
gain_vs_previous,0.060399,"[-0.1428151950513211, 0.25890701420147755]",2000,Descriptive paired client bootstrap conditiona...


## Incertitude et grands sinistres

Le bootstrap rééchantillonne les clients, pas les lignes indépendamment. L’intervalle est descriptif et conditionnel aux prédictions déjà ajustées. Un intervalle qui contient zéro ne prouve pas la supériorité du nouveau modèle.

In [6]:
n=pd.read_csv(R/'nested_predictions.csv');n['sq_error']=(n.cost-n.selected)**2
pos=n.loc[n.cost.gt(0)].sort_values('cost',ascending=False);k=int(np.ceil(.01*len(pos)))
print(f'Les {k} plus gros sinistres représentent {pos.head(k).sq_error.sum()/n.sq_error.sum():.1%} de l’erreur quadratique.')
fig,ax=plt.subplots();pd.DataFrame(nested['scores']).T.RMSE.plot.barh(ax=ax,color='#225ea8');ax.set(xlabel='RMSE',title='Comparaison sur les mêmes plis externes');plt.show()

Les 24 plus gros sinistres représentent 26.3% de l’erreur quadratique.


C:\Users\fourr\AppData\Local\Temp\ipykernel_30340\399770474.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();pd.DataFrame(nested['scores']).T.RMSE.plot.barh(ax=ax,color='#225ea8');ax.set(xlabel='RMSE',title='Comparaison sur les mêmes plis externes');plt.show()


## Export et reproductibilité

La soumission respecte le fichier officiel `prime_pred_sandbox.csv`. Les prédictions détaillées conservent séparément fréquence, gravité et produit. Le modèle sauvegardé se recharge sans exécuter un notebook. Les empreintes de données, code et versions sont enregistrées dans les rapports.

In [7]:
sub=pd.read_csv(ROOT/'submission.csv');sample=pd.read_csv(ROOT/'Data/prime_pred_sandbox.csv')
assert list(sub.columns)==list(sample.columns)==['index','pred']
np.testing.assert_array_equal(sub['index'],sample['index'])
assert np.isfinite(sub.pred).all() and sub.pred.ge(0).all()
print('Soumission valide :',len(sub),'lignes')
sub.head()

Soumission valide : 50000 lignes


,index,pred
0,50000,50.464029
1,50001,40.182099
2,50002,189.271520
3,50003,141.282667
4,50004,32.250457


In [8]:
kaggle_path=R/'kaggle_submissions.json'
if kaggle_path.exists():
    display(pd.DataFrame(json.loads(kaggle_path.read_text(encoding='utf-8'))['submissions']))
else:
    print('Score Kaggle à consulter après soumission. Aucun score local ne le remplace.')

Score Kaggle à consulter après soumission. Aucun score local ne le remplace.
